Linear regression model with normalized features are trained. The training results are reported at the end of the notebook.

In [1]:
import numpy as np 
import os 
import pandas as pd # 

In [2]:
df = pd.read_csv('match_data_300_tourns_modified.csv')

In [3]:
df.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage,tournament_id,date
0,Mark Allen,Ricky Walden,7,1580,1418,0.709992,0.599888,501,313,3345,...,45,25,1032,557,0,4,1.0,0.000000,1063,1.415318e+18
1,Stephen Maguire,Judd Trump,7,1563,1551,0.516401,0.507499,663,444,4995,...,122,79,1344,782,1,4,1.0,0.200000,1063,1.415059e+18
2,Mark Selby,Steve Davis,7,1592,1246,0.878716,0.703704,736,495,5330,...,8,2,581,286,4,1,0.0,0.800000,1063,1.415059e+18
3,Neil Robertson,Ali Carter,7,1546,1544,0.502734,0.501250,621,402,4581,...,11,5,876,499,4,0,0.0,1.000000,1063,1.415318e+18
4,Stuart Bingham,Ronnie O'Sullivan,7,1488,1663,0.275106,0.392337,744,462,5592,...,71,46,674,431,2,4,1.0,0.333333,1063,1.415146e+18


In [4]:
#Train test split
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, 
                                        test_size = 0.2,
                                        shuffle=False)

### Linear regression with features based on only player statistics
Here we pick only the statistical features for linear regression, elo ratings and match and frame win predictions from the elo ratings are omitted.

In [5]:
features_to_remove = ['player1', 'player2', 'player1_elo','player2_elo','elo_match_win_rate','elo_frame_win_rate',
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df_train.drop(columns=features_to_remove) 
y = df_train['win_percentage']

In [6]:
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline


In [7]:
kf = KFold(n_splits=5)

mae_list = []
train_sizes = []

for train_index, val_index in kf.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Validation size:  {len(val_index)}")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = LinearRegression()
    pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    mae_list.append(mae)
    train_sizes.append(len(train_index))

    print(f"Fold MAE: {mae} \n")

  Train size: 22092
  Validation size:  5524
Fold MAE: 0.21462768296318493 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.22227296590715007 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.22241918394666246 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.23080718199444242 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.23969887954919625 



In [8]:
avg_mae = np.average(mae_list)
print(f"Avg cross validation MAE:   {avg_mae:.4f}")

Avg cross validation MAE:   0.2260


### Linear regression with features based on player statistic differences
Intuitively, one can argue that differences in player statistics—such as match win ratio or frame win ratio—may be sufficient for the prediction scenario that we have. We confirm this claim by investigating corrleations between these feature and actual win percentage, as shown below.However, this approach overlooks the role of experience. For example, a player who has achieved the same win ratio as another but has played significantly more games may perform better due to their greater experience.

In [9]:
features_to_remove = ['player1', 'player2', 
                      'score1','score2','match_result','tournament_id']
df_ = df.drop(columns=features_to_remove) 
correlations = df_.corr()['win_percentage'].drop('win_percentage')
correlations = correlations.reindex(correlations.abs().sort_values(ascending=False).index)
print("Correlation with win percentage \n",correlations)

Correlation with win percentage 
 elo_match_win_rate          0.444988
elo_frame_win_rate          0.437116
player1_elo                 0.224803
player2_elo                -0.218702
p1_frames_won_3_years       0.201114
p1_frames_played_3_years    0.198630
p2_frames_won_3_years      -0.188409
p2_frames_played_3_years   -0.186126
p1_matches_won              0.165106
p1_matches_played           0.160169
p1_frames_won               0.158917
p1_frames_played            0.155919
p2_matches_won             -0.155563
p2_matches_played          -0.150385
p2_frames_won              -0.148245
p2_frames_played           -0.145045
p1_frames_won_1_year        0.139522
p1_frames_played_1_year     0.132124
p2_frames_won_1_year       -0.131668
p2_frames_played_1_year    -0.125887
date                       -0.002111
best_of                     0.001809
Name: win_percentage, dtype: float64


The statistics by themselves share absolute correlation coefficients of less than 0.2 with win pecentage.

In [10]:
dfm = df.copy()
dfm['p1_matches_win_ratio']=dfm['p1_matches_won']/df['p1_matches_played']
dfm['p2_matches_win_ratio']=dfm['p2_matches_won']/df['p2_matches_played']
dfm['p1_frames_win_ratio']=dfm['p1_frames_won']/df['p1_frames_played']
dfm['p2_frames_win_ratio']=dfm['p2_frames_won']/df['p2_frames_played']
dfm['p1_frames_win_ratio_1_year']=dfm['p1_frames_won_1_year']/df['p1_frames_played_1_year']
dfm['p2_frames_win_ratio_1_year']=dfm['p2_frames_won_1_year']/df['p2_frames_played_1_year']
dfm['p1_frames_win_ratio_3_years']=dfm['p1_frames_won_3_years']/df['p1_frames_played_3_years']
dfm['p2_frames_win_ratio_3_years']=dfm['p2_frames_won_3_years']/df['p2_frames_played_3_years']
dfm.fillna(0.5, inplace=True)
dfm['matches_win_ratio_diff']=dfm['p1_matches_win_ratio']-dfm['p2_matches_win_ratio']
dfm['frames_win_ratio_diff']=dfm['p1_frames_win_ratio']-dfm['p2_frames_win_ratio']
dfm['frames_win_ratio_diff_1_year']=dfm['p1_frames_win_ratio_1_year']-dfm['p2_frames_win_ratio_1_year']
dfm['frames_win_ratio_diff_3_years']=dfm['p1_frames_win_ratio_3_years']-dfm['p2_frames_win_ratio_3_years']
selected_features = ['matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years','win_percentage']
dfm_=dfm[selected_features]
correlations = dfm_.corr()['win_percentage'].drop('win_percentage')  # exclude self-correlation
correlations = correlations.reindex(correlations.abs().sort_values(ascending=False).index)
print("Correlation with win percentage \n",correlations)

Correlation with win percentage 
 matches_win_ratio_diff           0.346157
frames_win_ratio_diff            0.337987
frames_win_ratio_diff_3_years    0.333932
frames_win_ratio_diff_1_year     0.243961
Name: win_percentage, dtype: float64


The differences have much stronger correlation with win percentage.

In [11]:
print(df_train.columns)

Index(['player1', 'player2', 'best_of', 'player1_elo', 'player2_elo',
       'elo_match_win_rate', 'elo_frame_win_rate', 'p1_matches_played',
       'p1_matches_won', 'p1_frames_played', 'p1_frames_won',
       'p2_matches_played', 'p2_matches_won', 'p2_frames_played',
       'p2_frames_won', 'p1_frames_played_1_year', 'p1_frames_won_1_year',
       'p1_frames_played_3_years', 'p1_frames_won_3_years',
       'p2_frames_played_1_year', 'p2_frames_won_1_year',
       'p2_frames_played_3_years', 'p2_frames_won_3_years', 'score1', 'score2',
       'match_result', 'win_percentage', 'tournament_id', 'date'],
      dtype='object')


In [12]:
dfm = df_train.copy()

In [13]:
dfm['p1_matches_win_ratio']=dfm['p1_matches_won']/df['p1_matches_played']
dfm['p2_matches_win_ratio']=dfm['p2_matches_won']/df['p2_matches_played']
dfm['p1_frames_win_ratio']=dfm['p1_frames_won']/df['p1_frames_played']
dfm['p2_frames_win_ratio']=dfm['p2_frames_won']/df['p2_frames_played']
dfm['p1_frames_win_ratio_1_year']=dfm['p1_frames_won_1_year']/df['p1_frames_played_1_year']
dfm['p2_frames_win_ratio_1_year']=dfm['p2_frames_won_1_year']/df['p2_frames_played_1_year']
dfm['p1_frames_win_ratio_3_years']=dfm['p1_frames_won_3_years']/df['p1_frames_played_3_years']
dfm['p2_frames_win_ratio_3_years']=dfm['p2_frames_won_3_years']/df['p2_frames_played_3_years']



In [14]:
dfm.fillna(0.5, inplace=True)

In [15]:
dfm['matches_win_ratio_diff']=dfm['p1_matches_win_ratio']-dfm['p2_matches_win_ratio']
dfm['frames_win_ratio_diff']=dfm['p1_frames_win_ratio']-dfm['p2_frames_win_ratio']
dfm['frames_win_ratio_diff_1_year']=dfm['p1_frames_win_ratio_1_year']-dfm['p2_frames_win_ratio_1_year']
dfm['frames_win_ratio_diff_3_years']=dfm['p1_frames_win_ratio_3_years']-dfm['p2_frames_win_ratio_3_years']

In [16]:
selected_features = ['matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']

In [17]:
X = dfm[selected_features]
y = dfm['win_percentage']

In [18]:
kf = KFold(n_splits=5)

mae_list = []
train_sizes = []

for train_index, val_index in kf.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Validation size:  {len(val_index)}")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    mae_list.append(mae)
    train_sizes.append(len(train_index))

    print(f"Fold MAE: {mae} \n")

  Train size: 22092
  Validation size:  5524
Fold MAE: 0.22023289244218858 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.2237575388098332 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.2235171140886028 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.23252675921840404 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.23747323689794433 



In [19]:
avg_mae = np.average(mae_list)
print(f"Avg cross validation MAE:   {avg_mae:.4f}")

Avg cross validation MAE:   0.2275


The average MAEs are similar, suggesting that experience is not playing much of a factor.

### Linear regression on all available features

In [20]:
features_to_remove = ['player1', 'player2', 
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df_train.drop(columns=features_to_remove) 
y = df_train['win_percentage']

In [21]:
kf = KFold(n_splits=5)

mae_list = []
train_sizes = []

for train_index, val_index in kf.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Validation size:  {len(val_index)}")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = LinearRegression()
    pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    mae_list.append(mae)
    train_sizes.append(len(train_index))

    print(f"Fold MAE: {mae} \n")

  Train size: 22092
  Validation size:  5524
Fold MAE: 0.20686272352027762 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.2135852561400988 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.2147228289212999 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.22027558917901563 

  Train size: 22093
  Validation size:  5523
Fold MAE: 0.22612264938232024 



In [22]:
avg_mae = np.average(mae_list)
print(f"Avg cross validation MAE:   {avg_mae:.4f}")

Avg cross validation MAE:   0.2163


Average CV MAE with feature set 1: 0.2260

Average CV MAE with feature set 2: 0.2275

Average CV MAE with feature set 3: 0.2163